# RRL — Retrieval Reputation Layer Quickstart

Welcome to the **Retrieval Reputation Layer (RRL)** quickstart. RRL is a feedback-driven reranking layer that folds verified downstream outcomes back into document ranking using per-document **Beta counters with staleness decay**.

In this notebook, we will walk through:
1. **Ingesting** documents into the `CandidateStore`.
2. **Retrieving** documents using the RRL hybrid retriever (RRF + Thompson Sampling exploration).
3. **Updating counters** after observing downstream outcomes.
4. Using the **LangChain** and **LlamaIndex** integrations.

### 1. Basic In-Memory Store and Ingest
First, we initialize the candidate store and ingest some sample documents.

In [ ]:
from rrl.store import CandidateStore
from rrl.ingest import Ingester
from rrl.retriever import Retriever
from rrl.feedback import OutcomeSignals, update_counters, calculate_outcome

# Create store and ingester
store = CandidateStore()
ingester = Ingester()

# Ingest some sample documents
doc_ids = [
    ingester.ingest_document(store, "doc_1", "Python lists are dynamic arrays."),
    ingester.ingest_document(
        store, "doc_2", "Tuple objects in Python are immutable sequence types."
    ),
    ingester.ingest_document(store, "doc_3", "Set elements must be hashable and unique."),
]
print(f"Ingested {len(doc_ids)} documents.")

### 2. Retrieval
Now we initialize the retriever and retrieve candidates. Notice that the retriever returns the candidate objects, final scores, and raw similarity scores.

In [ ]:
# Initialize hybrid retriever
retriever = Retriever(store, weights=(0.30, 0.40, 0.10, 0.20))

# Retrieve candidates
query = "Are python tuples immutable?"
results = retriever.retrieve(query, top_k=2, explore=True)

for cand, score, sim in results:
    print(f"ID: {cand.id} | Score: {score:.3f} | Sim: {sim:.3f} | Content: '{cand.content}'")

### 3. Processing Feedback
After an outcome is observed (e.g. LLM judge rating, user behavior, unit tests), we update the store atomically.

In [ ]:
# Build similarity mapping of retrieved items to evaluate
retrieved_sims = {cand.id: sim for cand, _, sim in results}

# Record a successful outcome signal
signals = OutcomeSignals(s_gt=1.0)  # Downstream task succeeded
y = calculate_outcome(signals)

print(f"Calculated outcome outcome value: {y}")

# Update counters in the store
update_counters(store, retrieved_sims, y, signals=signals)

# Inspect updated Candidate counters
updated_cand = store.get_candidate("doc_2_chunk_0")
print(f"Updated doc_2 counters -> alpha: {updated_cand.alpha}, beta: {updated_cand.beta}")

### 4. LangChain Integration
We can use the `RRLRetriever` wrapper within any standard LangChain pipeline.

In [ ]:
from rrl.integrations.langchain import RRLRetriever

# Wrap our Retriever
lc_retriever = RRLRetriever(rrl_retriever=retriever)

# Retrieve documents using standard LangChain invoke
docs = lc_retriever.invoke("Set unique elements")
for doc in docs:
    print(
        f"Doc ID: {doc.metadata['id']} | Content: '{doc.page_content}' | Sim: {doc.metadata['rrl_sim']:.3f}"
    )

# Record feedback statelessly using the returned Document metadata
lc_retriever.record_feedback(docs, OutcomeSignals(s_gt=1.0))

### 5. LlamaIndex Integration
Similarly, we can use the `RRLLlamaIndexRetriever` wrapper within any LlamaIndex pipeline.

In [ ]:
from rrl.integrations.llama_index import RRLLlamaIndexRetriever

# Wrap our Retriever
li_retriever = RRLLlamaIndexRetriever(rrl_retriever=retriever)

# Retrieve nodes using standard LlamaIndex API
nodes = li_retriever.retrieve("Set unique elements")
for node in nodes:
    print(f"Node ID: {node.node.id_} | Content: '{node.node.text}' | Score: {node.score:.3f}")

# Record feedback statelessly using node metadata
li_retriever.record_feedback(nodes, OutcomeSignals(s_gt=1.0))